In [5]:
import os

import albumentations as A
import numpy as np
from albumentations.pytorch import ToTensorV2
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics.segmentation import DiceScore, JaccardIndex
from torchmetrics import Accuracy
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

ModuleNotFoundError: No module named 'albumentations'

In [ ]:
train_transform = A.Compose([
    A.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    # A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    A.Rotate(limit=60, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.5),
    ToTensorV2(),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = A.Compose([
    A.Resize(224,224),
    ToTensorV2(),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class SegDataset(Dataset):
    def __init__(self, image_paths, label_paths, transform=None):
        self.image_paths = image_paths
        self.label_paths = label_paths

        self.image_filenames = sorted(os.listdir(image_paths))
        self.label_filenames = sorted(os.listdir(label_paths))

        self.transform = transform

    def __getitem__(self, idx):
        image = cv2.imread(f'{self.image_paths}/{self.image_filenames[idx]}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(f'{self.label_paths}/{self.label_filenames[idx]}', cv2.IMREAD_GRAYSCALE)
        mask = mask / 255.0
        mask = mask.astype('float32')

        if self.transform:
            sample = self.transform(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']
        return image, mask

    def __len__(self):
        return len(self.image_filenames)

In [ ]:
class BCEAndDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.dice = smp.losses.DiceLoss(mode='binary')

        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        bce_loss = bce_loss.mean() * self.bce_weight

        dice_loss = self.dice(pred, target).mean() * self.dice_weight

        return bce_loss + dice_loss

In [ ]:
def create_loaders():
    train_dataset = SegDataset("tiff/train","tiff/train_labels",transform=train_transform)
    val_dataset = SegDataset("tiff/val","tiff/val_labels",transform=test_transform)
    test_dataset = SegDataset("tiff/test","tiff/test_labels",transform=test_transform)

    train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True, num_workers=32, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(dataset=val_dataset, batch_size=64, num_workers=32, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=64, num_workers=32, pin_memory=True, persistent_workers=True)

    return train_loader, val_loader, test_loader

In [ ]:
def create_model():
    model = smp.Unet(
        encoder_name="resnet50",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation="sigmoid"
    )
    model = model.to(device)

    return model

In [ ]:
def train(model, train_loader, val_loader, optimizer, criterion, scheduler, epochs):
    best_val_loss = float('inf')
    patience_counter = 0

    train_losses = []
    val_losses = []

    dice_metric = DiceScore(num_classes=1, threshold=0.5)
    iou_metric = JaccardIndex(num_classes=1, threshold=0.5)
    acc_metric = Accuracy(task='binary', threshold=0.5)

    model.train()
    for epoch in range(epochs):
        train_loss = 0
        for img, label in train_loader:
            img, label = img.to(device, non_blocking=True), label.to(device, non_blocking=True)

            optimizer.zero_grad()
            output = model(img)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        with torch.no_grad():
            val_loss = 0
            for img, label in val_loader:
                img, label = img.to(device, non_blocking=True), label.to(device, non_blocking=True)
                output = model(img)
                loss = criterion(output, label)
                val_loss += loss.item()
                dice_metric.update(output, label)
                iou_metric.update(output, label)
                acc_metric.update(output, label)

            val_loss /= len(val_loader)
            val_losses.append(val_loss)
            dice = dice_metric.compute()
            iou = iou_metric.compute()
            acc = acc_metric.compute()

            scheduler.step(val_loss)
            print(f"val loss: {val_loss:.4f}\ndice: {dice:.4f}\niou: {iou:.4f}\nacc: {acc:.4f}")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), f'model.pth')
                print("модель обновлена")
            else:
                patience_counter += 1

    return train_losses, val_losses


In [ ]:
def test(model, tes_loader, criterion):
    dice_metric = DiceScore(num_classes=1, threshold=0.5)
    iou_metric = JaccardIndex(num_classes=1, threshold=0.5)
    acc_metric = Accuracy(task='binary', threshold=0.5)

    with torch.no_grad():
        model.eval()
        dice_metric.reset()
        iou_metric.reset()
        acc_metric.reset()

        test_loss = 0
        for img, label in tes_loader:
            img, label = img.to(device, non_blocking=True), label.to(device, non_blocking=True)
            output = model(img)
            loss = criterion(output, label)

            dice_metric.update(output, label)
            iou_metric.update(output, label)
            acc_metric.update(output, label)
            test_loss += loss.item()

        test_loss /= len(tes_loader)
        dice = dice_metric.compute()
        iou = iou_metric.compute()
        acc = acc_metric.compute()

        print(f"test loss: {test_loss:.4f}\ndice: {dice:.4f}\niou: {iou:.4f}\nacc: {acc:.4f}")

In [ ]:
def train_graph(epochs, train_loss, val_loss):
    plt.figure(figsize=(10,5))

    plt.plot(epochs, train_loss, color="green", label="train loss")
    plt.plot(epochs, val_loss, color="yellow", label="val loss")

    plt.xlabel("эпоха")
    plt.ylabel("loss")
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

In [ ]:
def main():
    train_loader, val_loader, test_loader = create_loaders()
    model = create_model()
    criterion = BCEAndDiceLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
    train_loss, val_loss = train(model, train_loader, val_loader, optimizer, criterion, scheduler, epochs=20)
    train_graph(20, train_loss, val_loss)